# CIFAR Hydra on Google Colab

This notebook runs the MAEG3080 CIFAR-100 project on Colab with checkpoints stored in Google Drive.

**New session:** Runtime → **Change runtime type** → **T4 GPU** (or better if offered). Then run cells from the top.

If long training overheats your laptop, **run the heavy jobs here**; checkpoints and data live on Drive so they survive disconnects.

**Notebook file:** Use `colab/CIFAR_Hydra.ipynb` from this repo in your **new Colab project** (upload to Drive or Colab, or paste cells). One notebook is enough.

**Code bundle:** Run `colab/prepare_colab_bundle.sh` locally, then put `artifacts/cifar_hydra_project.tar.gz` into the Drive folder below—**avoid** Colab’s upload widget if it fails; use the Drive website.

**Fastest results:** After setup (mount → unpack → pip → env), run the **Quick results (SE-ResNet)** section — one cell runs baseline (if needed), short finetune, and official test.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import os
import pathlib
import shutil
import sys
import tarfile

WORK_ROOT = pathlib.Path('/content')
PROJECT_ROOT = WORK_ROOT / 'cifar-hydra'
MY_DRIVE = pathlib.Path('/content/drive/MyDrive')
DRIVE_ROOT = MY_DRIVE / 'Colab_CIFAR'
BUNDLE_NAME = 'cifar_hydra_project.tar.gz'

# If the file is not auto-detected, set the exact path here (uncomment and edit):
# MANUAL_BUNDLE = pathlib.Path('/content/drive/MyDrive/Colab_CIFAR/cifar_hydra_project.tar.gz')
MANUAL_BUNDLE = None


def resolve_bundle_path():
    if MANUAL_BUNDLE is not None and MANUAL_BUNDLE.is_file():
        return MANUAL_BUNDLE
    candidates = [
        DRIVE_ROOT / BUNDLE_NAME,
        MY_DRIVE / BUNDLE_NAME,
    ]
    for p in candidates:
        if p.is_file():
            return p
    if MY_DRIVE.is_dir():
        for p in MY_DRIVE.rglob(BUNDLE_NAME):
            if p.is_file():
                return p
    return None


BUNDLE_PATH = resolve_bundle_path()

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

if BUNDLE_PATH is None or not BUNDLE_PATH.exists():
    print('--- Debug: could not find', BUNDLE_NAME, '---')
    if MY_DRIVE.is_dir():
        try:
            top = sorted(p.name for p in MY_DRIVE.iterdir())
            print('My Drive (top level):', top[:40], '...' if len(top) > 40 else '')
        except OSError as e:
            print('Could not list My Drive:', e)
    if DRIVE_ROOT.is_dir():
        try:
            inside = sorted(p.name for p in DRIVE_ROOT.iterdir())
            print('Colab_CIFAR/:', inside)
        except OSError as e:
            print('Could not list Colab_CIFAR:', e)
    raise FileNotFoundError(
        f'Missing bundle. Expected name: {BUNDLE_NAME}\\n'
        f'1) On your PC: bash colab/prepare_colab_bundle.sh → uploads artifacts/{BUNDLE_NAME}\\n'
        f'2) In drive.google.com, upload that file into **My Drive/Colab_CIFAR/** (create the folder).\\n'
        f'   Or put it in **My Drive/** root and re-run.\\n'
        f'3) Wait for sync, then re-run this cell.\\n'
        f'4) Or set MANUAL_BUNDLE at the top of this cell to the full /content/drive/... path.'
    )

with tarfile.open(BUNDLE_PATH, 'r:gz') as archive:
    # filter= silences Python 3.12+ deprecation about safe tar extraction (3.14 default)
    if sys.version_info >= (3, 12):
        archive.extractall(PROJECT_ROOT, filter='data')
    else:
        archive.extractall(PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print('Using bundle:', BUNDLE_PATH)
print('Project root:', PROJECT_ROOT)
print('Drive root:', DRIVE_ROOT)


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


In [ ]:
import subprocess
import torch

subprocess.run(['nvidia-smi'], check=False)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import os
import pathlib
import sys
import torch

DATA_DIR = DRIVE_ROOT / 'data'
SAVE_DIR = DRIVE_ROOT / 'checkpoints'
OUTPUT_DIR = DRIVE_ROOT / 'outputs'

DATA_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ['PYTHON_BIN'] = sys.executable
os.environ['DATA_DIR'] = str(DATA_DIR)
os.environ['SAVE_DIR'] = str(SAVE_DIR)
os.environ['DEVICE'] = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['NUM_WORKERS'] = '2'
os.environ['BATCH_SIZE'] = '128'

print('DATA_DIR =', DATA_DIR)
print('SAVE_DIR =', SAVE_DIR)
print('DEVICE =', os.environ['DEVICE'])


## Quick results (SE-ResNet)

Runs **`scripts/quick_se_resnet_results.sh`**: trains the **4-epoch ~34% baseline** if `colab_repro_34pct_4epoch/best.pt` is missing, then **finetunes 30 epochs** from that checkpoint (`se_resnet_from_34pct`), then **`results.py`** on the official test.

Override (optional, in the next cell): `EPOCHS=60` for a stronger finetune; `BATCH_SIZE=128` if OOM.


In [ ]:
import os
import subprocess
import sys

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHON_BIN'] = sys.executable
env['DATA_DIR'] = str(DATA_DIR)
env['SAVE_DIR'] = str(SAVE_DIR)
env['DEVICE'] = os.environ['DEVICE']
env['NUM_WORKERS'] = '4'
env['TQDM_DISABLE'] = '1'
# Optional: env['EPOCHS'] = '60'
# Optional: env['BATCH_SIZE'] = '128'

process = subprocess.Popen(
    ['bash', 'scripts/quick_se_resnet_results.sh'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, ['bash', 'scripts/quick_se_resnet_results.sh'])


## Baseline repro (~34% on official test, 4 epochs)

This matches your historic **local** run (`checkpoints_cpu_4ep`): a **small SE-ResNet** (`base_width=32`, four stages × 2 blocks), **basic** augmentations, **no** Mixup/CutMix, **full 50k training targets**, and **`--eval-split test`** each epoch.

It is **not** the same as `hydra_ladder` / WRN-28-10 (which is why those runs look slower to climb early).

Run the next two cells **before** the long Hydra ladder if you want Colab to line up with that baseline. Checkpoints go to `SAVE_DIR/colab_repro_34pct_4epoch/`.

In [ ]:
import os
import subprocess
import sys

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHON_BIN'] = sys.executable
env['DATA_DIR'] = str(DATA_DIR)
env['SAVE_DIR'] = str(SAVE_DIR)
env['DEVICE'] = os.environ['DEVICE']
env['NUM_WORKERS'] = '2'
env['BATCH_SIZE'] = '64'
env['RUN_NAME'] = 'colab_repro_34pct_4epoch'
env['TQDM_DISABLE'] = '1'  # one line per epoch, no per-batch progress bars

process = subprocess.Popen(
    ['bash', 'scripts/repro_34pct_4epoch_baseline.sh'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, ['bash', 'scripts/repro_34pct_4epoch_baseline.sh'])

In [ ]:
import os
import pathlib
import subprocess
import sys

checkpoint_path = pathlib.Path(os.environ['SAVE_DIR']) / 'colab_repro_34pct_4epoch' / 'best.pt'
subprocess.run(
    [
        sys.executable,
        'results.py',
        '--checkpoint',
        str(checkpoint_path),
        '--data-dir',
        os.environ['DATA_DIR'],
        '--batch-size',
        '256',
        '--num-workers',
        '2',
        '--device',
        os.environ['DEVICE'],
    ],
    check=True,
)

## Iterate from ~34% baseline (optional)

Only if you **skipped** the **Quick results** cell above: same as `iterate_from_34pct.sh` — load **`colab_repro_34pct_4epoch/best.pt`**, finetune SE-ResNet. Then run **`results.py`** on `SAVE_DIR/se_resnet_from_34pct/best.pt`.

Defaults below: **30 epochs**, **batch 256** (override with `env['EPOCHS']` / `BATCH_SIZE`).


In [ ]:
import os
import subprocess
import sys

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHON_BIN'] = sys.executable
env['DATA_DIR'] = str(DATA_DIR)
env['SAVE_DIR'] = str(SAVE_DIR)
env['DEVICE'] = os.environ['DEVICE']
env['NUM_WORKERS'] = '4'
env['BATCH_SIZE'] = '256'
env['EPOCHS'] = '30'
env['TQDM_DISABLE'] = '1'
# Optional: env['EPOCHS'] = '60'
# Optional: env['RUN_NAME'] = 'se_resnet_from_34pct_v2'

process = subprocess.Popen(
    ['bash', 'scripts/iterate_from_34pct.sh'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, ['bash', 'scripts/iterate_from_34pct.sh'])


## Full ~50% pipeline (run-c → run-d → run-e → select-best → run-f → official test)

This matches **`scripts/run_fifty_percent_pipeline.sh`** on your machine: one long job that chains every stage and ends with `results.py`.

Optional environment overrides (uncomment in the next cell):

- `HYDRA_CDE_EPOCHS` / `RUN_F_EPOCHS` — defaults are **240** and **300** (full plan). Use `1` each for a quick smoke test.

The next cell sets **`TQDM_DISABLE=1`** so you see **epoch summaries only** (no per-batch tqdm—easier to copy).


In [ ]:
import os
import subprocess
import sys

# Optional smoke test (1 epoch per stage); omit these lines for the full ~50% plan.
# os.environ['HYDRA_CDE_EPOCHS'] = '1'
# os.environ['RUN_F_EPOCHS'] = '1'

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHON_BIN'] = sys.executable
env['TQDM_DISABLE'] = '1'  # quiet: epoch JSON only, no per-batch tqdm

process = subprocess.Popen(
    ['bash', 'scripts/run_fifty_percent_pipeline.sh'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, ['bash', 'scripts/run_fifty_percent_pipeline.sh'])


## Run Experiments (single stage)

Use this **only** if you want to run one ladder stage at a time instead of the **Full ~50% pipeline** section above.

Recommended order: `run-c`, `run-d`, `run-e`, `select-best`, then `run-f`.


In [ ]:
import os
import subprocess
import sys

run_mode = 'run-d'  # change to run-c, run-d, run-e, or run-f
best_run = 'hydra_run_d'  # only used for run-f

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['TQDM_DISABLE'] = '1'
if run_mode == 'run-f':
    env['BEST_RUN'] = best_run

process = subprocess.Popen(
    ['bash', 'scripts/hydra_ladder.sh', run_mode],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, process.args)


In [ ]:
import os
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        'scripts/select_best_run.py',
        os.path.join(os.environ['SAVE_DIR'], 'hydra_run_c'),
        os.path.join(os.environ['SAVE_DIR'], 'hydra_run_d'),
        os.path.join(os.environ['SAVE_DIR'], 'hydra_run_e'),
    ],
    check=True,
)


In [ ]:
import os
import pathlib
import subprocess
import sys

checkpoint_path = pathlib.Path(os.environ['SAVE_DIR']) / 'hydra_run_f_final' / 'best.pt'

subprocess.run(
    [
        sys.executable,
        'results.py',
        '--checkpoint',
        str(checkpoint_path),
        '--data-dir',
        os.environ['DATA_DIR'],
        '--batch-size',
        '256',
        '--num-workers',
        '2',
        '--device',
        os.environ['DEVICE'],
    ],
    check=True,
)
